In [1]:
import numpy as np
import pandas as pd
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from matplotlib import pyplot
import imblearn
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from sklearn import datasets
from sklearn.preprocessing import LabelBinarizer

In [2]:
df = pd.read_excel("/content/perfect_final_data_for_multilevel_analysis (1).xlsx")
df

,V001,V002,V012,V013,V024,V025,V106,V113,V130,V157,...,Number_of_children,Living_children,Age_of_husband,profession_of_husband,Birth_Interval,Drinking_water,Wealth_Status,Religion_status,BMI,Respondent_Age_first_birth
0,469,57,48,7,6,2,0,21,1,0,...,2,2,2,1,4,2,0,1,4,1
1,438,26,32,4,6,1,2,21,1,0,...,1,1,1,2,1,2,2,1,4,0
2,296,96,39,5,4,1,2,21,1,0,...,2,2,2,4,1,2,2,1,4,1
3,438,4,48,7,6,1,0,21,1,0,...,2,2,2,2,1,2,2,1,4,0
4,548,158,32,4,7,2,3,12,1,0,...,1,1,1,1,2,1,0,1,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8497,417,132,41,6,5,2,1,21,1,0,...,2,1,2,1,3,2,0,1,1,0
8498,190,11,43,6,3,1,0,11,1,0,...,1,1,1,3,3,1,1,1,1,1
8499,123,123,40,6,2,2,0,21,1,0,...,1,1,2,2,1,2,0,1,1,1
8500,581,64,30,4,7,2,1,21,2,0,...,2,2,1,2,2,2,0,0,1,1


In [3]:
df.drop(columns=['V001', 'V002', 'V012', 'V113', 'V130', 'V157', 'V158',
                 'V159', 'V190', 'V201', 'V212', 'V218', 'V221', 'V364', 'V445','V447', 'V501',
                 'V502', 'V701', 'V705', 'V730'], inplace=True)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8502 entries, 0 to 8501
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   V013                        8502 non-null   int64  
 1   V024                        8502 non-null   int64  
 2   V025                        8502 non-null   int64  
 3   V106                        8502 non-null   int64  
 4   V404                        8502 non-null   int64  
 5   V714                        8502 non-null   int64  
 6   BMI_fixed                   8502 non-null   float64
 7   Media_acccess               8502 non-null   int64  
 8   husband_education           8502 non-null   int64  
 9   contraceptive_use           8502 non-null   int64  
 10  Number_of_children          8502 non-null   int64  
 11  Living_children             8502 non-null   int64  
 12  Age_of_husband              8502 non-null   int64  
 13  profession_of_husband       8502 

In [5]:
df.drop(columns=['BMI_fixed'], inplace=True)

In [6]:
#Min-Max Scaling (FT2)
from sklearn.preprocessing import MinMaxScaler
FT2 = MinMaxScaler()
FT2_data = FT2.fit_transform(df)
FT2_df = pd.DataFrame(FT2_data, columns=df.columns)
FT2_df.head()

,V013,V024,V025,V106,V404,V714,Media_acccess,husband_education,contraceptive_use,Number_of_children,Living_children,Age_of_husband,profession_of_husband,Birth_Interval,Drinking_water,Wealth_Status,Religion_status,BMI,Respondent_Age_first_birth
0,1.000000,0.714286,1.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.5,1.0,1.0,0.25,0.666667,0.50,0.0,1.0,1.0,1.0
1,0.500000,0.714286,0.0,0.666667,0.0,1.0,1.0,0.666667,0.0,0.0,0.5,0.5,0.50,0.166667,0.50,1.0,1.0,1.0,0.0
2,0.666667,0.428571,0.0,0.666667,0.0,0.0,1.0,1.000000,1.0,0.5,1.0,1.0,1.00,0.166667,0.50,1.0,1.0,1.0,1.0
3,1.000000,0.714286,0.0,0.000000,0.0,0.0,1.0,0.666667,0.0,0.5,1.0,1.0,0.50,0.166667,0.50,1.0,1.0,1.0,0.0
4,0.500000,0.857143,1.0,1.000000,0.0,0.0,0.0,0.333333,0.0,0.0,0.5,0.5,0.25,0.333333,0.25,0.0,1.0,1.0,1.0


In [7]:

y = FT2_df.BMI  # use bmi1 as the target variable
x = FT2_df.drop('BMI', axis=1)  # use the other columns as features

In [8]:
x  # display the feature columns

,V013,V024,V025,V106,V404,V714,Media_acccess,husband_education,contraceptive_use,Number_of_children,Living_children,Age_of_husband,profession_of_husband,Birth_Interval,Drinking_water,Wealth_Status,Religion_status,Respondent_Age_first_birth
0,1.000000,0.714286,1.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.5,1.0,1.0,0.25,0.666667,0.50,0.0,1.0,1.0
1,0.500000,0.714286,0.0,0.666667,0.0,1.0,1.0,0.666667,0.0,0.0,0.5,0.5,0.50,0.166667,0.50,1.0,1.0,0.0
2,0.666667,0.428571,0.0,0.666667,0.0,0.0,1.0,1.000000,1.0,0.5,1.0,1.0,1.00,0.166667,0.50,1.0,1.0,1.0
3,1.000000,0.714286,0.0,0.000000,0.0,0.0,1.0,0.666667,0.0,0.5,1.0,1.0,0.50,0.166667,0.50,1.0,1.0,0.0
4,0.500000,0.857143,1.0,1.000000,0.0,0.0,0.0,0.333333,0.0,0.0,0.5,0.5,0.25,0.333333,0.25,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8497,0.833333,0.571429,1.0,0.333333,0.0,0.0,0.0,0.000000,1.0,0.5,0.5,1.0,0.25,0.500000,0.50,0.0,1.0,0.0
8498,0.833333,0.285714,0.0,0.000000,0.0,0.0,1.0,0.000000,0.0,0.0,0.5,0.5,0.75,0.500000,0.25,0.5,1.0,1.0
8499,0.833333,0.142857,1.0,0.000000,0.0,0.0,0.0,0.000000,1.0,0.0,0.5,1.0,0.50,0.166667,0.50,0.0,1.0,1.0
8500,0.500000,0.857143,1.0,0.333333,1.0,0.0,1.0,0.333333,1.0,0.5,1.0,0.5,0.50,0.333333,0.50,0.0,0.0,1.0


In [9]:
y = LabelEncoder().fit_transform(y)  # change target labels into numbers
oversample = SMOTE()  # create SMOTE to balance classes
x, y = oversample.fit_resample(x, y)  # make class counts more balanced
counter = Counter(y)  # count samples in each class
for k, v in counter.items():  # loop through each class count
  per = v / len(y) * 100  # calculate class percentage
  print('class=%d, count=%d, percentage=%.3f%%' % (k, v, per))  # print class distribution

class=3, count=4469, percentage=25.000%
class=2, count=4469, percentage=25.000%
class=1, count=4469, percentage=25.000%
class=0, count=4469, percentage=25.000%


In [10]:
from sklearn.model_selection import train_test_split  # import the train-test split tool
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=101)  # split data into training and testing sets
x_train.head()  # preview the training features
x_train.shape  # check the training data shape

(12513, 18)

In [11]:
x_test.head()  # preview the testing features
x_test.shape  # check the testing data shape

(5363, 18)

In [12]:
# logistic regression
from sklearn.linear_model import LogisticRegression  # import logistic regression
Lr = LogisticRegression()  # create the model
Lr.fit(x_train, y_train)  # train the model on the training data

# Generate predictions for both test and training data
predictions = Lr.predict(x_test)  # predict labels for the test set
train_predictions = Lr.predict(x_train)  # predict labels for the training set

from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools

# Train performance
print("=== Training Performance ===")  # print training results heading
print(classification_report(y_train, train_predictions, digits=4))  # show training metrics
print(confusion_matrix(y_train, train_predictions))  # show training confusion matrix
# test performance
print("=== Test Performance ===")  # print test results heading
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = Lr.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities for ROC work

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # measure agreement between true and predicted labels

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(Lr, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

=== Training Performance ===
              precision    recall  f1-score   support

           0     0.4446    0.6042    0.5122      3120
           1     0.3116    0.1876    0.2342      3113
           2     0.3427    0.2153    0.2645      3112
           3     0.4462    0.6259    0.5210      3168

    accuracy                         0.4093     12513
   macro avg     0.3863    0.4083    0.3830     12513
weighted avg     0.3866    0.4093    0.3837     12513

[[1885  541  305  389]
 [1239  584  513  777]
 [ 671  476  670 1295]
 [ 445  273  467 1983]]
=== Test Performance ===
              precision    recall  f1-score   support

           0     0.4280    0.5819    0.4932      1349
           1     0.3039    0.1851    0.2301      1356
           2     0.3249    0.2085    0.2540      1357
           3     0.4312    0.6072    0.5043      1301

    accuracy                         0.3933      5363
   macro avg     0.3720    0.3957    0.3704      5363
weighted avg     0.3713    0.3933    0

In [13]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier  # import decision tree model
dtree = DecisionTreeClassifier()  # create the model
dtree.fit(x_train, y_train)  # train the model
predictions = dtree.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = dtree.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(dtree, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.6709    0.6679    0.6694      1349
           1     0.4981    0.4727    0.4851      1356
           2     0.4237    0.4171    0.4203      1357
           3     0.6349    0.6818    0.6575      1301

    accuracy                         0.5585      5363
   macro avg     0.5569    0.5599    0.5581      5363
weighted avg     0.5559    0.5585    0.5569      5363

[[901 152 158 138]
 [167 641 416 132]
 [164 387 566 240]
 [111 107 196 887]]
Cross Validation Scores are [0.57941834 0.62248322 0.59004474 0.56991051 0.58780761 0.59619687
 0.59373251 0.58142138 0.56183548 0.59149412 0.60626398 0.60346756
 0.59563758 0.57718121 0.58612975 0.5917226  0.59429211 0.58925574
 0.58869614 0.60212647 0.61073826 0.60067114 0.59451902 0.58165548
 0.60178971 0.58557047 0.5775042  0.58030218 0.57246782 0.58477896]
Average Cross Validation score :0.5899705053524772


In [14]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier  # import random forest model
rfc = RandomForestClassifier(n_estimators=100)  # create the model with 100 trees
rfc.fit(x_train, y_train)  # train the model
predictions = rfc.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = rfc.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(rfc, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.8406    0.8132    0.8267      1349
           1     0.5812    0.7257    0.6455      1356
           2     0.6102    0.4569    0.5225      1357
           3     0.8028    0.8324    0.8174      1301

    accuracy                         0.7056      5363
   macro avg     0.7087    0.7070    0.7030      5363
weighted avg     0.7076    0.7056    0.7016      5363

[[1097  147   43   62]
 [  53  984  279   40]
 [ 102  471  620  164]
 [  53   91   74 1083]]
Cross Validation Scores are [0.72818792 0.74440716 0.73154362 0.72986577 0.73713647 0.73881432
 0.7313934  0.7302742  0.71516508 0.73195299 0.73545861 0.75447427
 0.72259508 0.73042506 0.73993289 0.74496644 0.72747622 0.73195299
 0.74706212 0.72579743 0.73489933 0.74105145 0.73489933 0.74832215
 0.73993289 0.71420582 0.74706212 0.73419138 0.73866816 0.72635702]
Average Cross Validation score :0.7346157224665504


In [15]:
# K- nearest neighbor
from sklearn.neighbors import KNeighborsClassifier  # import KNN model
KN = KNeighborsClassifier()  # create the model
KN.fit(x_train, y_train)  # train the model
predictions = KN.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = KN.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(KN, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.6502    0.9370    0.7677      1349
           1     0.5585    0.2603    0.3551      1356
           2     0.6532    0.5122    0.5741      1357
           3     0.7121    0.9431    0.8115      1301

    accuracy                         0.6599      5363
   macro avg     0.6435    0.6631    0.6271      5363
weighted avg     0.6428    0.6599    0.6250      5363

[[1264   32   24   29]
 [ 440  353  311  252]
 [ 217  230  695  215]
 [  23   17   34 1227]]
Cross Validation Scores are [0.68232662 0.69071588 0.68456376 0.68791946 0.69407159 0.70190157
 0.68942361 0.70229435 0.69110241 0.70565193 0.68903803 0.71644295
 0.69686801 0.69798658 0.69854586 0.68568233 0.6927812  0.69054281
 0.70117515 0.68326805 0.68400447 0.71029083 0.69966443 0.69519016
 0.68288591 0.69407159 0.68550644 0.70341354 0.69390039 0.70453274]
Average Cross Validation score :0.6945254211896593


In [16]:
# GaussianNB
from sklearn.naive_bayes import GaussianNB  # import Gaussian Naive Bayes model
NB = GaussianNB()  # create the model
NB.fit(x_train, y_train)  # train the model
predictions = NB.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = NB.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(NB, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.4291    0.6279    0.5098      1349
           1     0.3731    0.2190    0.2760      1356
           2     0.3516    0.1842    0.2418      1357
           3     0.4373    0.6326    0.5171      1301

    accuracy                         0.4134      5363
   macro avg     0.3978    0.4159    0.3862      5363
weighted avg     0.3973    0.4134    0.3846      5363

[[847 214 117 171]
 [553 297 205 301]
 [340 180 250 587]
 [234 105 139 823]]
Cross Validation Scores are [0.42225951 0.42114094 0.40883669 0.42058166 0.42505593 0.4189038
 0.41969782 0.41242306 0.4045887  0.423615   0.41834452 0.41442953
 0.42002237 0.39149888 0.4295302  0.4189038  0.42809177 0.41130386
 0.41410185 0.42137661 0.44239374 0.41778523 0.40324385 0.41946309
 0.41442953 0.41107383 0.39619474 0.423615   0.42697258 0.40123111]
Average Cross Validation score :0.416703639300658


In [17]:
# support vector machine
from sklearn.svm import LinearSVC  # import linear SVM model
classifier = LinearSVC()  # create the model
classifier.fit(x_train, y_train)  # train the model
y_predict = classifier.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, y_predict, digits=4))  # show test metrics
print(confusion_matrix(y_test, y_predict))  # show test confusion matrix
y_score = classifier.fit(x_train, y_train).decision_function(x_test)  # get decision scores

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, y_predict)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(classifier, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.4142    0.6731    0.5128      1349
           1     0.3432    0.1195    0.1772      1356
           2     0.3327    0.1341    0.1912      1357
           3     0.4020    0.6649    0.5010      1301

    accuracy                         0.3947      5363
   macro avg     0.3730    0.3979    0.3456      5363
weighted avg     0.3727    0.3947    0.3437      5363

[[908 136  94 211]
 [635 162 160 399]
 [378 120 182 677]
 [271  54 111 865]]
Cross Validation Scores are [0.41275168 0.40715884 0.39932886 0.41219239 0.41498881 0.3909396
 0.4029099  0.40123111 0.39115837 0.41522104 0.39205817 0.40939597
 0.41051454 0.39261745 0.40044743 0.41107383 0.41634024 0.39955232
 0.4029099  0.41186346 0.41610738 0.41498881 0.40771812 0.4082774
 0.39709172 0.40268456 0.39059877 0.40235031 0.40906547 0.38332401]
Average Cross Validation score :0.4042286824180104


In [18]:
# AdaBoostClassifier
from sklearn.ensemble import AdaBoostClassifier  # import AdaBoost model
abc = AdaBoostClassifier(n_estimators=50, learning_rate=1)  # create the model
model = abc.fit(x_train, y_train)  # train the model
y_pred = model.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, y_pred, digits=4))  # show test metrics
print(confusion_matrix(y_test, y_pred))  # show test confusion matrix
y_score = abc.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(abc, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.5265    0.5604    0.5429      1349
           1     0.5038    0.3953    0.4430      1356
           2     0.3034    0.1968    0.2387      1357
           3     0.4236    0.6457    0.5116      1301

    accuracy                         0.4473      5363
   macro avg     0.4393    0.4495    0.4340      5363
weighted avg     0.4393    0.4473    0.4331      5363

[[756 207 176 210]
 [291 536 213 316]
 [222 251 267 617]
 [167  70 224 840]]
Cross Validation Scores are [0.46979866 0.46252796 0.44183445 0.45917226 0.45917226 0.45861298
 0.47118075 0.47341914 0.45103525 0.44823727 0.45693512 0.46029083
 0.4541387  0.46308725 0.46420582 0.47706935 0.48349189 0.45271405
 0.45942921 0.46950196 0.4614094  0.44966443 0.46420582 0.49384787
 0.4647651  0.46756152 0.45886961 0.45942921 0.47285954 0.46278679]
Average Cross Validation score :0.46304181496406854


In [19]:
# GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier  # import gradient boosting model
GB = GradientBoostingClassifier()  # create the model
model = GB.fit(x_train, y_train)  # train the model
predictions = model.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = GB.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(GB, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.7050    0.6820    0.6933      1349
           1     0.5607    0.7839    0.6538      1356
           2     0.4926    0.2682    0.3473      1357
           3     0.6261    0.6849    0.6542      1301

    accuracy                         0.6038      5363
   macro avg     0.5961    0.6048    0.5871      5363
weighted avg     0.5956    0.6038    0.5863      5363

[[ 920  207   59  163]
 [  47 1063  207   39]
 [ 160  503  364  330]
 [ 178  123  109  891]]
Cross Validation Scores are [0.60961969 0.62807606 0.60682327 0.60626398 0.6163311  0.61129754
 0.6144376  0.58365976 0.59149412 0.61891438 0.59228188 0.62416107
 0.60458613 0.60178971 0.60514541 0.62360179 0.61052043 0.59373251
 0.61779519 0.61387801 0.60794183 0.6196868  0.60290828 0.63422819
 0.60626398 0.61800895 0.60212647 0.60380526 0.60548405 0.59820929]
Average Cross Validation score :0.6091024246286149


In [20]:
# XGBoost
from numpy import loadtxt  # import loadtxt from NumPy
from xgboost import XGBClassifier  # import XGBoost model
XGB = XGBClassifier()  # create the model
model = XGB.fit(x_train, y_train)  # train the model
predictions = model.predict(x_test)  # predict labels for the test set
from sklearn.metrics import classification_report, confusion_matrix  # import evaluation tools
print(classification_report(y_test, predictions, digits=4))  # show test metrics
print(confusion_matrix(y_test, predictions))  # show test confusion matrix
y_score = XGB.fit(x_train, y_train).predict_proba(x_test)  # get prediction probabilities

from sklearn.metrics import cohen_kappa_score  # import kappa score
cohen_kappa_score(y_test, predictions)  # calculate agreement score

from sklearn.model_selection import RepeatedStratifiedKFold  # import repeated stratified k-fold
from sklearn.model_selection import cross_val_score, KFold  # import cross-validation tools
kf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)  # define the cross-validation plan
scores = cross_val_score(XGB, x, y, cv=kf)  # evaluate the model with cross-validation
print("Cross Validation Scores are {}".format(scores))  # print all cross-validation scores
print("Average Cross Validation score :{}".format(scores.mean()))  # print the average score

              precision    recall  f1-score   support

           0     0.7939    0.7739    0.7838      1349
           1     0.5574    0.7308    0.6324      1356
           2     0.5570    0.3854    0.4556      1357
           3     0.7626    0.7802    0.7713      1301

    accuracy                         0.6662      5363
   macro avg     0.6677    0.6676    0.6608      5363
weighted avg     0.6666    0.6662    0.6594      5363

[[1044  180   53   72]
 [  69  991  263   33]
 [ 134  489  523  211]
 [  68  118  100 1015]]
Cross Validation Scores are [0.69574944 0.69127517 0.69183445 0.69742729 0.6935123  0.69742729
 0.68438724 0.68214885 0.68494684 0.691662   0.68903803 0.70302013
 0.67729306 0.6901566  0.69127517 0.69407159 0.66648013 0.69054281
 0.6922216  0.69390039 0.69407159 0.70581655 0.68400447 0.71252796
 0.69519016 0.69463087 0.68326805 0.68998321 0.68270845 0.69725797]
Average Cross Validation score :0.691260990073724


In [21]:
# Bagging
from sklearn.ensemble import BaggingClassifier
BC=BaggingClassifier()
model=BC.fit(x_train, y_train)
predictions = model.predict(x_test)
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,predictions,digits=4))
print(confusion_matrix(y_test,predictions))
y_score = BC.fit(x_train, y_train).predict_proba(x_test)

from sklearn.metrics import cohen_kappa_score
cohen_kappa_score(y_test,predictions)

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import  cross_val_score,KFold
kf = RepeatedStratifiedKFold(n_splits = 10, n_repeats=3, random_state=1)
scores = cross_val_score(BC, x, y, cv = kf)
print("Cross Validation Scores are {}".format(scores))
print("Average Cross Validation score :{}".format(scores.mean()))

              precision    recall  f1-score   support

           0     0.7199    0.7583    0.7386      1349
           1     0.5428    0.6217    0.5796      1356
           2     0.4958    0.3928    0.4383      1357
           3     0.7489    0.7563    0.7526      1301

    accuracy                         0.6308      5363
   macro avg     0.6269    0.6323    0.6273      5363
weighted avg     0.6255    0.6308    0.6258      5363

[[1023  158   95   73]
 [ 131  843  324   58]
 [ 179  446  533  199]
 [  88  106  123  984]]
Cross Validation Scores are [0.65771812 0.66610738 0.65268456 0.65939597 0.66666667 0.65100671
 0.67207611 0.65528819 0.65193061 0.6726357  0.66331096 0.65883669
 0.66331096 0.64932886 0.66778523 0.6689038  0.6530498  0.6536094
 0.66200336 0.67431449 0.66219239 0.66219239 0.64709172 0.66219239
 0.6442953  0.66107383 0.64465585 0.65920537 0.65584779 0.64689424]
Average Cross Validation score :0.6588534957291599
